# Structured Outputs & Constrainted Decoding

Ask an LLM for JSON. Get JSON most of the time. In production, "most" is the problem. Constrained decoding turns "most" into "always" by editing the logits before sampling.

## Problem Definition

Three layers exist in 2026.

### Promopting

Ask nicely. "Return only the JSON object." Works for ~80% on frontier models, less on samller ones.

### Native structured output APIs

OpenAI `response_format`, Anthropic tool use, Gemini JSON mode.
Reliabe on supperted schemas, Vendor-locked.

### Constrainted Decoding

**Modify the logits at every generation step** so the model cannot emit invalid tokens. 100% valid by construction.


## Basic Concept

### How constrained decoding works

The LLM produces a logit vector over the full vocabulary, a logit processor sits between the model and the sampler. It computes which token are valid given the current position in the target grammer - JSON Schema, regex, context-free grammer, And sets the logits of all invalid tokens to negative infinity.

### Implements vendor

* **Outlines**.     Complies JSON schema or regex into a finite-state machine.
* **XGrammer/llguidance**.  Context-free grammer engine.
* **vLLM guided decoding**
* **Instructor** Pydantic-base wrapper over any LLM. Retries on validation failure.

## Counterintuitive result

Constrained decoding is often faster that unconstained generation.

1. Shrinks the next-token search space
2. Skip token generation entirely for forced tokens, eg "name": "



# Build your Own

In [1]:
import math
import random
import re
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../../00_Common").resolve()))
from user_tools import SectionPrinter

PHONE_REGEX = fr"^\d{3}-\d{3}-\d{4}$"

class PhoneFSM:
    def __init__(self):
        self.accept_state = 12

    def valid_next(self, state):
        if state in (0, 1, 2, 4, 5, 6, 8, 9, 10, 11):
            return list("0123456789")
        if state in (3, 7):
            return ["-"]
        if state == 12:
            return []
        
        return ValueError(f"unknown state {state}")

    def transition(self, state, ch):
        if ch not in self.valid_next(state):
            return None
        return state + 1

    def is_accept(self, state):
        return state == self.accept_state

def softmax(logits):
    finite = [x for x in logits if x != float("-inf")]
    if not finite:
        return [0.0] * len(logits)
    m = max(finite)
    exps = [math.exp(x - m) if x != float("-inf") else 0.0 for x in logits]
    z = sum(exps)
    return [e / z for e in exps]

def sample(probs, rng):
    r = rng.random()
    acc = 0.0
    for i, p in enumerate(probs):
        acc += p
        if r <= acc:
            return i
    return len(probs) - 1

def mask_logits(logits, valid_indices):
    return [logits[i] if i in valid_indices else float("-inf") for i in range(len(logits))]

def fake_llm_logits(alphabet, rng):
    return [rng.gauss(0.0, 1.5) for _ in alphabet]

def generate_constrained(alphabet, fsm, seed):
    rng = random.Random(seed)
    alphabet_idx = {ch: i for i, ch in enumerate(alphabet)}
    state = 0
    out = ""
    while not fsm.is_accept(state):
        logits = fake_llm_logits(alphabet, rng)
        valid_chars = fsm.valid_next(state)
        if not valid_chars:
            break
        valid_ids = {alphabet_idx[ch] for ch in valid_chars}
        masked = mask_logits(logits, valid_ids)
        probs = softmax(masked)
        next_ch = alphabet[sample(probs, rng)]
        out += next_ch
        state = fsm.transition(state, next_ch)
        if state is None:
            break
    return out

def generate_unconstrained(alphabet, max_len, seed):
    rng = random.Random(seed)
    out = ""
    for _ in range(max_len):
        logits = fake_llm_logits(alphabet, rng)
        probs = softmax(logits)
        next_ch = alphabet[sample(probs, rng)]
        out += next_ch
    return out

with SectionPrinter("Generate a phone number"):
    ALPHABET = "0123456789-"
    fsm = PhoneFSM()
    print(generate_constrained(ALPHABET, fsm, 42))
    print(generate_unconstrained(ALPHABET, 12, 42))

==================Generate a phone number===================
006-994-2188
0063--422188


# Outlines for JSON Sechema

In [2]:
from pydantic import BaseModel
from typing import Literal

import outlines
from transformers import AutoModelForCausalLM, AutoTokenizer

class Review(BaseModel):
    sentiment: Literal["positive", "negative", "neutral"]
    confidence: float
    evidence_span: str

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
hf_model = AutoModelForCausalLM.from_pretrained(model_name)
hf_tokenizer = AutoTokenizer.from_pretrained(model_name)
if hf_tokenizer.pad_token is None:
    hf_tokenizer.pad_token = hf_tokenizer.eos_token

model = outlines.from_transformers(hf_model, hf_tokenizer)
generator = outlines.Generator(model, Review)

prompt = "Classify: 'The wait staff was attentive and the food arrived hot.'"
result = generator(prompt, max_new_tokens=120)
review = Review.model_validate_json(result)

with SectionPrinter("Outlines JSON Schema"):
    print(review)


W0806 21:09:46.874000 87782 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
W0806 21:09:46.915000 87782 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0806 21:09:46.934000 87782 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

====================Outlines JSON Schema====================
sentiment='positive' confidence=0.9543617823956225 evidence_span='[0, 11]'


## Native vendor APIS

In [3]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../../00_Common").resolve()))
from user_tools import SectionPrinter, load_project_env

from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.chat_models import init_chat_model

load_project_env()

prompt = "Classify: 'The wait staff was attentive and the food arrived hot.'"

agent = create_agent(
    model=init_chat_model(
        "deepseek:deepseek-chat",
        extra_body={"thinking": {"type": "disabled"}},
    ),
    response_format=ToolStrategy(
        Review,
        tool_message_content="Return only the JSON object.",
    ),
)

response = agent.invoke({"messages": [{"role": "user", "content": prompt}]})

with SectionPrinter("Structured output APIs"):
    print(response["messages"][-1].content)
    print(response.get("structured_response"))


/Users/keyficller/Documents/AIEngineering/.venv/lib/python3.14/site-packages/langchain/chat_models/base.py:496: UserWarning: WARNING! response_format is not default parameter.
                response_format was transferred to model_kwargs.
                Please confirm that response_format is what you intended.
  return _init_chat_model_helper(


InvalidUpdateError: Expected dict, got Classify: 'The wait staff was attentive and the food arrived hot.'
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/INVALID_GRAPH_NODE_RETURN_VALUE